# 🎙️ Haru-4B: Live Voice Portal & Quantized Model Testing

Test **[Jayanth-MKV/Haru-4B-GGUF](https://huggingface.co/Jayanth-MKV/Haru-4B-GGUF)** and **[higurashi-ai/Haru-4B](https://huggingface.co/higurashi-ai/Haru-4B)** on **Google Colab** (Free T4 GPU).

### ⚡ Ensure GPU is Selected:
Click **Runtime** > **Change runtime type** > Select **T4 GPU** > **Save**.

In [ ]:
# Step 1: Verify GPU
!nvidia-smi

---
## 🧪 Part 1: Test Our Quantized Model Backbone (Jayanth-MKV/Haru-4B-GGUF)

Installs pre-compiled GPU `llama-cpp-python` and runs our 2.6 GB model on the GPU:

In [ ]:
# 1. Install pre-compiled GPU llama-cpp, CUDA runtime, and cuBLAS (takes ~5 seconds)
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 huggingface_hub

# 2. Pre-load CUDA runtime & cuBLAS so libllama.so finds them
import ctypes, glob, os

# Load CUDA Runtime
for path in glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib/libcudart.so*") + glob.glob("/usr/local/cuda*/lib64/libcudart.so*"):
    try:
        ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
        break
    except Exception:
        pass

# Load cuBLAS
for path in glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cublas/lib/libcublas.so*"):
    try:
        ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
        break
    except Exception:
        pass

# 3. Download our Quantized GGUF Model (2.6 GB)
from huggingface_hub import hf_hub_download
print("Downloading qwen-haru-q4_k_m.gguf from Jayanth-MKV/Haru-4B-GGUF...")
model_path = hf_hub_download(repo_id="Jayanth-MKV/Haru-4B-GGUF", filename="qwen-haru-q4_k_m.gguf")
print("Downloaded successfully to:", model_path)

# 4. Run high-speed GPU inference (n_gpu_layers=-1 offloads all 32 layers to VRAM)
from llama_cpp import Llama
print("Loading model into T4 GPU VRAM...")
llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=2048, verbose=False)

prompt = "Hello! How are you doing today? Let's have a conversation."
print(f"\nPrompt: {prompt}\nGenerating response with our quantized model...")
output = llm(prompt, max_tokens=128, stop=["\n\n", "User:"])
print("\n--- Output ---")
print(output["choices"][0]["text"].strip())

---
## 🎙️ Part 2: Launch the Full-Duplex Live Voice Portal (Port 8998 with Preloaded Voice)

Downloads a default voice preset so **Start Conversation is automatically enabled** with 0 manual uploads, then starts the live portal:

In [ ]:
# 1. Clone Haru & Install Dependencies into /content/Haru/haru
!rm -rf /content/Haru
!git clone https://github.com/higurashi-ai/Haru.git /content/Haru
%cd /content/Haru/haru
!pip install -q -e ".[share]"

# 2. Install default English reference voice (enables 'Start conversation' automatically!)
import urllib.request, os
os.makedirs("/content/Haru/haru/haru/assets/voices", exist_ok=True)
voice_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
urllib.request.urlretrieve(voice_url, "/content/Haru/haru/haru/assets/voices/en.flac")
print("Default voice installed: 'Start conversation' is now active out of the box!")

# 3. Launch Haru Full-Duplex Web Server with Public HTTPS Tunnel
# --warmup-frames 0 loads the model in 3 seconds
# --share creates a secure public link (https://xxxx.gradio.live) for your microphone
!python -m haru.server --share --warmup-frames 0 --host 0.0.0.0 --port 8998